### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [1]:
!pip install langchain langchain-core langchain-community pypdf pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install langchain-community
import os

os.makedirs("/content/data", exist_ok=True)
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/tmp/ipykernel_3091/3280871835.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [19]:
### Fun to Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):

    all_pages = []

    pdf_dir = Path(pdf_directory)   # gives Path-object
    pdf_files = list(pdf_dir.glob("**/*.pdf"))   # list of Path-objects for all pdf-files.

    print(f"Found {len(pdf_files)} PDF files in Folder, going to process these..")


    for p in pdf_files:
            loader = PyPDFLoader(str(p))
            pages_doc_ds = loader.load() # load all pages in Document-DS of p PDFfile (matadata + page content)

            # Add extra information to metadata
            for page in pages_doc_ds:
                page.metadata['source_file'] = p.name
                page.metadata['file_type'] = 'pdf'

            all_pages.extend(pages_doc_ds)

            print(f"  ✓ Loaded {len(pages_doc_ds)} pages")



    print(f"\nTotal documents loaded: {len(all_pages)}")
    return all_pages




all_pdf_documents = process_all_pdfs("/content/data")    # calling this fun

Found 2 PDF files in Folder, going to process these..
  ✓ Loaded 1 pages
  ✓ Loaded 3 pages

Total documents loaded: 4


1 document  =  1 page of 1 PDF

In [20]:
### Fun for Text splitting to create Chunks

def create_chunk (documents,chunk_size=1000,chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(chunks)} chunks")


    return chunks

In [24]:
chunks=create_chunk(all_pdf_documents)
print(chunks)

Split 4 documents into 5 chunks
[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-01-21T15:20:59+05:30', 'author': 'Shikha Chaudhary [MU - Jaipur]', 'moddate': '2026-01-21T15:20:59+05:30', 'source': '/content/data/AssignmentAWS.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'AssignmentAWS.pdf', 'file_type': 'pdf'}, page_content='Faculty of Science, Technology and Architecture \nSchool of Computer Science and Engineering \nDepartment of Information Technology \nINT 3250 Advanced Data Structures (PE6) \nVI Semester IT (Common to all sections) \nAssignment- 1                                       M.M: 20 \n \n \n1. Design red black tree of the following sequence \n90,130,143,151,123,133,164,157 \nDelete the following node from the above tree \n 151, 157, 143 \n \n2. Perform an extract min operation on the heap below \n \n3.  In the following Fibonacci heap decrease the key value 

### Use ChromaDB for embeddings and vector store:

In [25]:
!pip install chromadb
import chromadb

In [27]:
clint = chromadb.Client()
collection = clint.get_or_create_collection("docs")

In [28]:
!pip install sentence_transformers
from sentence_transformers import SentenceTransformer

In [29]:
m = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [30]:
i = 1
for c in chunks:
  collection.add(
      documents=[c.page_content],
      embeddings=[m.encode(c.page_content)],
      metadatas=[c.metadata],
      ids=[str(i)]
  )
  i=i+1

In [31]:
collection.peek()

{'ids': ['1', '2', '3', '4', '5'],
 'embeddings': array([[-0.05976133,  0.12478022, -0.00731947, ..., -0.10416912,
         -0.038996  ,  0.03939844],
        [ 0.05488931,  0.01476577, -0.02463914, ...,  0.03778877,
         -0.06133717,  0.01702921],
        [ 0.00630396, -0.05032212, -0.03550008, ...,  0.06043293,
          0.00106522,  0.05267261],
        [ 0.01502579,  0.02371687, -0.0897411 , ...,  0.08196656,
         -0.05912803, -0.01009377],
        [-0.04760569,  0.03534198, -0.05063309, ...,  0.06819799,
         -0.02914735, -0.01197759]]),
 'documents': ['Faculty of Science, Technology and Architecture \nSchool of Computer Science and Engineering \nDepartment of Information Technology \nINT 3250 Advanced Data Structures (PE6) \nVI Semester IT (Common to all sections) \nAssignment- 1                                       M.M: 20 \n \n \n1. Design red black tree of the following sequence \n90,130,143,151,123,133,164,157 \nDelete the following node from the above tree \n 15

In [32]:
# fun for result simmlear to quary

def simm(q, k):
  q_emb = m.encode(q)
  result = collection.query(
      query_embeddings= q_emb,
      n_results= k
  )
  return result

In [33]:
a = simm("what is crc?", 2)
a['documents']

[['Using data from Q8, assume BER = (10^{-5}). \n1. Compute probability of frame error \n2. Estimate retransmissions in Go-Back-N \n3. Estimate retransmissions in Selective Repeat \n4. Compare efficiency numerically \n \nQ8. Signal Attenuation and Power \nA signal with initial power 10 mW passes through 3 sections of cable with attenuation: \n• 3 dB \n• 10 dB \n• 7 dB \n1. Calculate total attenuation \n2. Find final received power \n3. If minimum detectable power is 0.1 µW, determine if communication is possible \nQ9. Framing Techniques \nCompare: \n• Byte stuffing \n• Bit stuffing \nExplain advantages and failure cases. \n \nQ10. CRC vs Checksum Mathematical Strength \nWhy is CRC mathematically stronger than checksum? Explain in terms of polynomial arithmetic. \n \nQ11. Sliding Window Sequence Numbering \nIf window size = 7: \n1. What is minimum sequence number field size for Go-Back-N? \n2. What is minimum sequence number field size for Selective Repeat? \n3. Justify mathematically.'

In [34]:
!pip install openai
!pip install langchain-openai langchain

In [ ]:
#import os
#os.environ["XAI_API_KEY"] = "gsk_MUWiXDmkDq0nwxKtjtQoWODZn5bnR1g3sPB7fI6RsRnUh"

In [ ]:
from openai import OpenAI
clint = OpenAI(
    api_key = "gsk_MUWiXDmkDq0nwxKtjtQoWGdODZn5bnR1g3sPB7fI6RsRnUh",
    base_url= "https://api.groq.com/openai/v1"
)

In [37]:
def chat_with_llm(q, c, m="llama-3.1-8b-instant"):
  prompt = f"""
          Use the following context to answer the question concisely.
          context : {c['documents']}
          question : {q}
          """


  responce = clint.chat.completions.create(
      model = m,
      messages=[{
          "role": "user",
          "content": prompt
      }]
  )

  return responce.choices[0].message.content

In [39]:
# input
que = input("Hello! How may I help you? :  ")

context = simm(que, 2)
print(chat_with_llm(que,context))


Hello! How may I help you? :  give me few example questions on crc.
Here are a few example questions based on the context:

1. **CRC Q5:**
Given data: 1101011011110001
Generator polynomial: (G(x) = x5 + x4 + x2 + x + 1)
   - Find CRC remainder
   - Give transmitted codeword
   - Show receiver verification process

2. **CRC Q10:**
- What is the difference in mathematical strength between CRC and checksum in terms of polynomial arithmetic?
- Provide an example of a polynomial where CRC performs better than checksum for error detection.

3. **CRC Q7:**
- Describe a scenario where CRC is more efficient than checksum for error detection in a network protocol.
- Provide a comparison of the error detection capabilities of CRC and checksum for a given data packet.

These questions test your understanding of CRC concepts, polynomial arithmetic, and error detection capabilities.
